In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")


CUDA available: True
GPU count: 1
GPU name: NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
yaml_path = "../Orthomosaics/datasets/dataset_rgb_filip_aabb_overlap100_non_augmented/dataset.yaml"

In [ ]:
#!/usr/bin/env python3
yaml_path = "/home/samuel/test/MasterThesis/Orthomosaics/datasets/dataset_NIR_RED_NGRDI_EXTENDED_aabb_pngs/dataset.yaml"
from ultralytics import YOLO
import os

# Use a larger model
model = YOLO("yolo26n.pt")  # or yolo12m.pt if you have 24GB+ VRAM

results = model.train(
    data=yaml_path,
    epochs=80,        # More epochs with better hardware
    imgsz=1024,        
    batch=32,          # Much larger batch (adjust based on VRAM)
    patience=30,       
    freeze=0,               
    mixup=0.0,
    
    #IMPORTANT MOSAIC IS OFF NOW
    #mosaic=0.0,
    
    #copy_paste=0.3,    # Add copy-paste augmentation
    close_mosaic=15,   
    hsv_h=0, hsv_s=0, hsv_v=0,   
    degrees=0, scale=0, translate=0, shear=0, perspective=0,  
    optimizer='AdamW', # Explicit optimizer
    workers=8,
    #cache=True,         # Cache images in RAM (~10GB for your set) → faster epochs
    #compile=True,  
    dropout=0.0,        # YOLO handles reg well via augmentation
    erasing=0.0,        # Random erasing helps with occlusion
    box=5.0,
    cls=1.0,            # Single class → cls loss less important
    dfl=1.5,
    save=True,
    save_period=25,
    val=True,
    device=0,
    seed=42,            # Reproducibility
    verbose=True
)

print("Training complete.")

FileNotFoundError: [Errno 2] No such file or directory: 'yolo12s-obb.pt'

: 

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO
from ultralytics import RTDETR
import numpy as np

box_type = "aabb"  # "obb" or "aabb"
test_image = "/content/dataset/dataset_NIR_RED_NGRDI_EXTENDED_GLOBAL_AUGMENTED_true_AABB_pngs/images/test"
images = os.listdir(test_image)

model = YOLO("/content/runs/detect/train-5/weights/best.pt")
run: int = 0
prediction_output = Path("/content/predictions/nrn/nrn_aug_yolo12s_global")

if prediction_output.exists():
    run = len(list(prediction_output.glob("run*")))

prediction_output = prediction_output / f"run{run}"
while prediction_output.exists():
    run += 1
    prediction_output = prediction_output.parent / f"run{run}"
prediction_output.mkdir(parents=True, exist_ok=True)

preprocess_times = []
inference_times  = []
postprocess_times = []


for img_name in images:
    img_path = os.path.join(test_image, img_name)
    if "xml" in img_name:
        continue
    results = model.predict(
        source=img_path,
        imgsz=1024,
        conf=0.01,  # match COCO eval
        iou=0.5,
        save_txt=True,
        save_conf=True,
        exist_ok=True
    )

        # Ultralytics stores speed as a dict on each result
    for result in results:
        preprocess_times.append(result.speed["preprocess"])
        inference_times.append(result.speed["inference"])
        postprocess_times.append(result.speed["postprocess"])

    prediction_file = img_name.replace(".png", ".txt")

    for result in results:
        img_w, img_h = result.orig_shape[1], result.orig_shape[0]

        if box_type == "obb":
            if result.obb is None or len(result.obb) == 0:
                continue

            for obb in result.obb:
                cls  = int(obb.cls[0])
                conf = float(obb.conf[0])

                # 4 corner points in pixel coords [[x1,y1], [x2,y2], [x3,y3], [x4,y4]]
                corners = obb.xyxyxyxy[0].tolist()

                pts_norm = [(x / img_w, y / img_h) for x, y in corners]
                flat = [coord for pt in pts_norm for coord in pt]

                with open(prediction_output / prediction_file, "a") as f:
                    f.write(
                        f"{cls} {conf:.4f} "
                        f"{' '.join(f'{v:.6f}' for v in flat)}\n"
                    )

        elif box_type == "aabb":
            if result.boxes is None or len(result.boxes) == 0:
                continue

            for box in result.boxes:
                cls  = int(box.cls[0])
                conf = float(box.conf[0])

                # xywh returns [cx, cy, w, h] in pixel coords
                cx, cy, w, h = box.xywh[0].tolist()

                # Normalise to [0, 1]
                cx_n = cx / img_w
                cy_n = cy / img_h
                w_n  = w  / img_w
                h_n  = h  / img_h

                with open(prediction_output / prediction_file, "a") as f:
                    f.write(
                        f"{cls} {conf:.4f} "
                        f"{cx_n:.6f} {cy_n:.6f} {w_n:.6f} {h_n:.6f}\n"
                    )

    print(f"Inference complete for {img_name}. Results saved.")

avg_pre   = np.mean(preprocess_times)
avg_infer = np.mean(inference_times)
avg_post  = np.mean(postprocess_times)
avg_total = avg_pre + avg_infer + avg_post
fps       = 1000.0 / avg_total

print(f"\n========== INFERENCE SPEED ==========")
print(f"Avg preprocess  : {avg_pre:.1f} ms")
print(f"Avg inference   : {avg_infer:.1f} ms")
print(f"Avg postprocess : {avg_post:.1f} ms")
print(f"Avg total       : {avg_total:.1f} ms")
print(f"FPS             : {fps:.2f}")

print("All inference complete. Results saved.")


image 1/1 /home/samuel/test/MasterThesis/Orthomosaics/datasets/dataset_NIR_RED_NGRDI_EXTENDED_GLOBAL_AUGMENTED_true_AABB_pngs/images/test/large_local_data_final_tile_3_12__translated_250x_250y_rotated_45.png: 1024x1024 (no detections), 14.9ms
Speed: 5.4ms preprocess, 14.9ms inference, 1.5ms postprocess per image at shape (1, 3, 1024, 1024)
Results saved to /home/samuel/MasterThesis/runs/detect/predict
0 label saved to /home/samuel/MasterThesis/runs/detect/predict/labels
Inference complete for large_local_data_final_tile_3_12__translated_250x_250y_rotated_45.png. Results saved.

image 1/1 /home/samuel/test/MasterThesis/Orthomosaics/datasets/dataset_NIR_RED_NGRDI_EXTENDED_GLOBAL_AUGMENTED_true_AABB_pngs/images/test/large_local_data_final_tile_11_7__rotated45.png: 1024x1024 2 potatoess, 14.7ms
Speed: 4.4ms preprocess, 14.7ms inference, 17.0ms postprocess per image at shape (1, 3, 1024, 1024)
Results saved to /home/samuel/MasterThesis/runs/detect/predict
1 label saved to /home/samuel/Mast

In [ ]:
from pathlib import Path
import numpy as np
from scipy.optimize import linear_sum_assignment
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict


def compute_iou(box1, box2):
    """
    YOLO format: [x_center, y_center, width, height]
    """

    b1_x1 = box1[0] - box1[2] / 2
    b1_y1 = box1[1] - box1[3] / 2
    b1_x2 = box1[0] + box1[2] / 2
    b1_y2 = box1[1] + box1[3] / 2

    b2_x1 = box2[0] - box2[2] / 2
    b2_y1 = box2[1] - box2[3] / 2
    b2_x2 = box2[0] + box2[2] / 2
    b2_y2 = box2[1] + box2[3] / 2

    ix1 = max(b1_x1, b2_x1)
    iy1 = max(b1_y1, b2_y1)
    ix2 = min(b1_x2, b2_x2)
    iy2 = min(b1_y2, b2_y2)

    intersection = max(0, ix2 - ix1) * max(0, iy2 - iy1)

    area1 = box1[2] * box1[3]
    area2 = box2[2] * box2[3]

    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0.0


def match_predictions_optimal(preds, gts, iou_threshold=0.5):
    """
    Hungarian matching for object detection.
    """

    if len(preds) == 0 or len(gts) == 0:
        return set(), set()

    iou_matrix = np.zeros((len(preds), len(gts)), dtype=np.float32)

    for i, (cls_p, conf_p, box_p) in enumerate(preds):
        for j, (cls_g, box_g) in enumerate(gts):

            # Class mismatches are set to -1.0 so the Hungarian algorithm
            # never prefers them over a valid same-class match (even IoU=0).
            if cls_p != cls_g:
                iou_matrix[i, j] = -1.0
                continue

            iou_matrix[i, j] = compute_iou(box_p, box_g)

    row_ind, col_ind = linear_sum_assignment(-iou_matrix)

    matched_preds = set()
    matched_gts = set()

    for r, c in zip(row_ind, col_ind):

        if iou_matrix[r, c] >= iou_threshold:
            matched_preds.add(r)
            matched_gts.add(c)

    return matched_preds, matched_gts


def compute_pr_curve(
    all_predictions_by_image,
    all_gts_per_image,
    iou_threshold=0.5,
    pr_conf_threshold=0.001,
    num_thresholds=200
):
    """
    Sweep confidence thresholds from 1.0 down to pr_conf_threshold,
    computing precision and recall at each point.

    all_predictions_by_image is a dict: image_id -> [(cls, conf, box), ...]
    pre-grouped for O(1) lookup instead of scanning the flat list each time.
    """

    thresholds = np.linspace(1.0, pr_conf_threshold, num_thresholds)

    precisions = []
    recalls = []

    for conf_thresh in thresholds:

        TP = 0
        FP = 0
        FN = 0

        for image_id, gts in all_gts_per_image.items():

            # O(1) lookup — no scanning the full prediction list
            preds = [
                (cls, conf, box)
                for cls, conf, box in all_predictions_by_image.get(image_id, [])
                if conf >= conf_thresh
            ]

            matched_preds, matched_gts = match_predictions_optimal(
                preds,
                gts,
                iou_threshold
            )

            TP += len(matched_preds)
            FP += len(preds) - len(matched_preds)
            FN += len(gts) - len(matched_gts)

        precision = TP / (TP + FP + 1e-9)
        recall    = TP / (TP + FN + 1e-9)

        precisions.append(precision)
        recalls.append(recall)

    precisions = np.array(precisions)
    recalls    = np.array(recalls)

    # Sort ascending by recall
    order      = np.argsort(recalls)
    recalls    = recalls[order]
    precisions = precisions[order]

    # Precision envelope: for each recall point, take the max precision
    # seen at any recall >= that point (right-to-left accumulation).
    precisions = np.maximum.accumulate(precisions[::-1])[::-1]

    map50 = np.trapezoid(precisions, recalls)

    return map50, recalls, precisions


def plot_pr_curve(
    recalls,
    precisions,
    map50,
    save_path,
    model_name="Model"
):

    fig, ax = plt.subplots(figsize=(8, 6))

    ax.plot(
        recalls,
        precisions,
        linewidth=2,
        label=f"{model_name} (mAP@0.5 = {map50:.3f})"
    )

    ax.fill_between(recalls, precisions, alpha=0.15)

    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.05])
    ax.set_xlabel("Recall", fontsize=13)
    ax.set_ylabel("Precision", fontsize=13)
    ax.set_title("Precision-Recall Curve", fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="lower left")

    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)

    print(f"PR curve saved to: {save_path}")


def evaluate(
    pred_dir,
    gt_dir,
    iou_threshold=0.5,
    conf_threshold=0.3,
    pr_conf_threshold=0.001,
    model_name="Model",
    output_dir="."
):

    pred_dir   = Path(pred_dir)
    gt_dir     = Path(gt_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    gt_files = list(gt_dir.glob("*.txt"))

    TP = 0
    FP = 0
    FN = 0

    confidences = []

    # Pre-grouped by image_id for O(1) lookup in compute_pr_curve
    all_predictions_by_image = defaultdict(list)
    all_gts_per_image        = {}

    total_gt = 0

    # =========================
    # Load GT + Predictions
    # =========================

    for gt_file in gt_files:

        image_id  = gt_file.stem
        pred_file = pred_dir / gt_file.name

        # -------------------------
        # Load GT
        # -------------------------

        gts = []

        for line in gt_file.read_text().splitlines():

            parts = line.strip().split()

            if len(parts) < 5:
                continue

            cls = int(parts[0])
            box = list(map(float, parts[1:5]))
            gts.append((cls, box))

        total_gt += len(gts)
        all_gts_per_image[image_id] = gts

        # -------------------------
        # Load Predictions
        # -------------------------

        preds = []

        if pred_file.exists():

            for line in pred_file.read_text().splitlines():

                parts = line.strip().split()

                if len(parts) < 6:
                    continue

                cls  = int(parts[0])
                conf = float(parts[1])
                box  = list(map(float, parts[2:6]))

                preds.append((cls, conf, box))

                # ALL predictions stored for PR curve (no conf filter yet)
                all_predictions_by_image[image_id].append((cls, conf, box))

        # -------------------------
        # Fixed-threshold metrics
        # -------------------------

        preds_filtered = [
            (cls, conf, box)
            for cls, conf, box in preds
            if conf >= conf_threshold
        ]

        matched_preds, matched_gts = match_predictions_optimal(
            preds_filtered,
            gts,
            iou_threshold
        )

        TP += len(matched_preds)
        FP += len(preds_filtered) - len(matched_preds)
        FN += len(gts) - len(matched_gts)

        confidences.extend(conf for _, conf, _ in preds_filtered)

    # =========================
    # Fixed threshold metrics
    # =========================

    precision = TP / (TP + FP + 1e-9)
    recall    = TP / (TP + FN + 1e-9)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)
    accuracy  = TP / (TP + FP + FN + 1e-9)
    avg_conf  = np.mean(confidences) if confidences else 0.0

    # =========================
    # PR Curve + mAP
    # =========================

    map50, recalls, precisions = compute_pr_curve(
        all_predictions_by_image,
        all_gts_per_image,
        iou_threshold,
        pr_conf_threshold
    )

    # =========================
    # Print metrics
    # =========================

    print("\n========== RESULTS ==========")
    print(f"Total GT        : {total_gt}")
    print(f"TP              : {TP}")
    print(f"FP              : {FP}")
    print(f"FN              : {FN}")
    print(f"Precision       : {precision:.4f}  (conf >= {conf_threshold})")
    print(f"Recall          : {recall:.4f}  (conf >= {conf_threshold})")
    print(f"F1 Score        : {f1:.4f}  (conf >= {conf_threshold})")
    print(f"Accuracy        : {accuracy:.4f}")
    print(f"Avg Confidence  : {avg_conf:.4f}")
    print(f"mAP@0.5         : {map50:.4f}")

    # =========================
    # Plot PR
    # =========================

    plot_pr_curve(
        recalls,
        precisions,
        map50,
        save_path=output_dir / f"{model_name}_pr_curve.png",
        model_name=model_name
    )


if __name__ == "__main__":

    evaluate(
        pred_dir="/content/predictions/nrn/nrn_aug_yolo26s_obb/run0",

        gt_dir="/content/dataset/dataset_NIR_RED_NGRDI_EXTENDED_LOCAL_AUGMENTED_true_AABB_pngs/labels/test",

        iou_threshold=0.5,

        conf_threshold=0.3,

        pr_conf_threshold=0.01,

        model_name="YOLO26s",

        output_dir="/content/output/nrn/yolo26s_nrnlocal_aug"
    )


========== RESULTS ==========
Total GT        : 3539
TP              : 3313
FP              : 214
FN              : 226
Precision       : 0.9393  (conf >= 0.3)
Recall          : 0.9361  (conf >= 0.3)
F1 Score        : 0.9377  (conf >= 0.3)
Accuracy        : 0.8828
Avg Confidence  : 0.8977
mAP@0.5         : 0.9550
PR curve saved to: /home/samuel/test/MasterThesis/GLOBAL/evaluation_plots/YOLO12s_pr_curve.png


In [4]:
from pathlib import Path
from collections import Counter


# ============================================================
# CONFIG
# ============================================================

# Path to your labels folder
labels_root = Path("/home/samuel/test/MasterThesis/Orthomosaics/datasets/dataset_NIR_RED_NGRDI_EXTENDED_GLOBAL_AUGMENTED_true_AABB_pngs/labels")

# Which subset to analyze
subset_name = "test"


# ============================================================
# FUNCTIONS
# ============================================================

def count_yolo_instances(label_dir: Path):
    """
    Count YOLO object instances in all .txt files.
    YOLO format:
        class_id x_center y_center width height
    """

    class_counter = Counter()

    total_objects = 0
    total_files = 0
    empty_files = 0

    txt_files = sorted(label_dir.rglob("*.txt"))

    for txt_file in txt_files:

        total_files += 1

        with open(txt_file, "r") as f:
            lines = [line.strip() for line in f.readlines() if line.strip()]

        if len(lines) == 0:
            empty_files += 1
            continue

        for line in lines:

            parts = line.split()

            # Skip malformed lines
            if len(parts) < 5:
                print(f"Skipping malformed line in {txt_file}: {line}")
                continue

            class_id = int(parts[0])

            class_counter[class_id] += 1
            total_objects += 1

    return {
        "total_files": total_files,
        "empty_files": empty_files,
        "total_objects": total_objects,
        "class_counter": class_counter,
    }


def calculate_dataset_split(labels_root: Path):

    subsets = ["train", "val", "test"]

    split_info = {}
    total_files = 0

    for subset in subsets:

        subset_dir = labels_root / subset

        if not subset_dir.exists():
            split_info[subset] = 0
            continue

        count = len(list(subset_dir.rglob("*.txt")))

        split_info[subset] = count
        total_files += count

    split_percentages = {}

    for subset, count in split_info.items():

        if total_files == 0:
            split_percentages[subset] = 0.0
        else:
            split_percentages[subset] = 100.0 * count / total_files

    return split_info, split_percentages, total_files


# ============================================================
# RUN ANALYSIS
# ============================================================

subset_dir = labels_root / subset_name

if not subset_dir.exists():
    raise FileNotFoundError(f"Subset directory not found: {subset_dir}")

stats = count_yolo_instances(subset_dir)

split_info, split_percentages, total_files = calculate_dataset_split(
    labels_root
)


# ============================================================
# PRINT RESULTS
# ============================================================

print("========== YOLO INSTANCE COUNT ==========")
print(f"Subset:            {subset_name}")
print(f"Total label files: {stats['total_files']}")
print(f"Empty label files: {stats['empty_files']}")
print(f"Total objects:     {stats['total_objects']}")

print("\nObjects per class:")

for class_id, count in sorted(stats["class_counter"].items()):
    print(f"  Class {class_id}: {count}")

print("\n========== DATASET SPLIT ==========")
print(f"Total files across splits: {total_files}\n")

for subset in ["train", "val", "test"]:

    print(
        f"{subset:<5}: "
        f"{split_info[subset]:>5} files "
        f"({split_percentages[subset]:6.2f}%)"
    )

print("===================================")

========== YOLO INSTANCE COUNT ==========
Subset:            test
Total label files: 2213
Empty label files: 894
Total objects:     3539

Objects per class:
  Class 0: 3539

========== DATASET SPLIT ==========
Total files across splits: 11054

train:  6638 files ( 60.05%)
val  :  2203 files ( 19.93%)
test :  2213 files ( 20.02%)
